## Cálculo Tabla Provincias INE

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Definición de rutas de ficheros del Registro de la Propiedad 
export_ine_path = Path(r"C:\Users\LorenaMéndez\Desktop\Analisis_Precios_vivienda\Data - IPV INE")
indices_file = export_ine_path / "IPV - Export 25171 - Indices.xlsx"
ponderaciones_file = export_ine_path / "IPV - Export 25174 - Ponderaciones.xlsx"

In [ ]:
## 1. Carga de datos del indices de IPV del INE
# Carga del fichero de indices del INE
df_ipv_ine_prev = pd.read_excel(indices_file, sheet_name="tabla-25171-indice")

## Aplicar transformaciones
# primera columna (la que contiene Nacional, Andalucía, etc.)
col0 = df_ipv_ine_prev.columns[0]

# detectar filas cabecera de CCAA:
# aquí usamos una columna numérica cualquiera
mask_ccaa = df_ipv_ine_prev["2025T4"].isna()

# crear columna CCAA
df_ipv_ine_prev["ccaa"] = df_ipv_ine_prev[col0].where(mask_ccaa)

# rellenar hacia abajo
df_ipv_ine_prev["ccaa"] = df_ipv_ine_prev["ccaa"].ffill()

# mover CCAA al inicio
df_ipv_ine_prev.insert(0, "ccaa", df_ipv_ine_prev.pop("ccaa"))

# Renombrar columnas
df_ipv_ine_prev.rename(
    columns={df_ipv_ine_prev.columns[1]: "medida"},
    inplace=True
)

# Eliminar filas con valores NaN
df_ipv_ine_prev = df_ipv_ine_prev[
    df_ipv_ine_prev["ccaa"] != df_ipv_ine_prev["medida"]
].reset_index(drop=True)

# Calculo de la columna cod_ccaa
df_ipv_ine_prev["cod_ccaa"] = np.where(
    df_ipv_ine_prev["ccaa"] == "Nacional",
    99,
    df_ipv_ine_prev["ccaa"].str.extract(r"^(\d{2})")[0]
).astype(int)

# mover cod_ccaa al inicio
df_ipv_ine_prev.insert(0, "cod_ccaa", df_ipv_ine_prev.pop("cod_ccaa"))

#Visualizar los resultados
df_ipv_ine_prev

In [ ]:
# Transponer los datos (convertir las columnas de tiempo en filas)
df_ipv_ine = df_ipv_ine_prev.melt(
    id_vars=["cod_ccaa", "ccaa", "medida"],
    var_name="periodo",
    value_name="ipv"
)


# Calculo de la columna "fecha"
# extraer año y trimestre
df_ipv_ine["ejercicio"] = df_ipv_ine["periodo"].str[:4].astype(int)
df_ipv_ine["trim"] = df_ipv_ine["periodo"].str[-1].astype(int)
# mapa trimestre -> mes
mapa_mes = {
    1: 1,
    2: 4,
    3: 7,
    4: 10
}
# crear fecha
df_ipv_ine["fecha"] = pd.to_datetime({
    "year": df_ipv_ine["ejercicio"],
    "month": df_ipv_ine["trim"].map(mapa_mes),
    "day": 1
})

# Substituir caracteres en blanco en la columna "medida"
df_ipv_ine["medida"] = df_ipv_ine["medida"].str.strip()
df_ipv_ine["medida"] = df_ipv_ine["medida"].str.replace(" ", "_")

# Calculo del campo "id"
df_ipv_ine["id"] = (
    df_ipv_ine["cod_ccaa"].astype(str)
    + "_"
    + df_ipv_ine["medida"]
    + "_"
    + df_ipv_ine["periodo"]
)

# Ordenar el DF
df_ipv_ine = df_ipv_ine[['id', 'cod_ccaa', 'ccaa', 'medida', 'periodo', 'fecha', 'ejercicio','trim', 'ipv']]

# Visualizar los datos
df_ipv_ine

In [ ]:
#df_ipv_ine.columns

In [ ]:
## ***************************************************************************

In [ ]:
## 2. Carga de datos de la variación anual del IPV del INE
# Carga del fichero de indices del INE
df_ipv_var_anual_ine_prev = pd.read_excel(indices_file, sheet_name="tabla-25171-varanual")

## Aplicar transformaciones
# primera columna (la que contiene Nacional, Andalucía, etc.)
col0 = df_ipv_var_anual_ine_prev.columns[0]

# detectar filas cabecera de CCAA:
# aquí usamos una columna numérica cualquiera
mask_ccaa = df_ipv_var_anual_ine_prev["2025T4"].isna()

# crear columna CCAA
df_ipv_var_anual_ine_prev["ccaa"] = df_ipv_var_anual_ine_prev[col0].where(mask_ccaa)

# rellenar hacia abajo
df_ipv_var_anual_ine_prev["ccaa"] = df_ipv_var_anual_ine_prev["ccaa"].ffill()

# mover CCAA al inicio
df_ipv_var_anual_ine_prev.insert(0, "ccaa", df_ipv_var_anual_ine_prev.pop("ccaa"))

# Renombrar columnas
df_ipv_var_anual_ine_prev.rename(
    columns={df_ipv_var_anual_ine_prev.columns[1]: "medida"},
    inplace=True
)

# Eliminar filas con valores NaN
df_ipv_var_anual_ine_prev = df_ipv_var_anual_ine_prev[
    df_ipv_var_anual_ine_prev["ccaa"] != df_ipv_var_anual_ine_prev["medida"]
].reset_index(drop=True)

# Calculo de la columna cod_ccaa
df_ipv_var_anual_ine_prev["cod_ccaa"] = np.where(
    df_ipv_var_anual_ine_prev["ccaa"] == "Nacional",
    99,
    df_ipv_var_anual_ine_prev["ccaa"].str.extract(r"^(\d{2})")[0]
).astype(int)

# mover cod_ccaa al inicio
df_ipv_var_anual_ine_prev.insert(0, "cod_ccaa", df_ipv_var_anual_ine_prev.pop("cod_ccaa"))

# Visualizar los resultados
df_ipv_var_anual_ine_prev

In [ ]:
# Transponer los datos (convertir las columnas de tiempo en filas)
df_ipv_var_anual_ine = df_ipv_var_anual_ine_prev.melt(
    id_vars=["cod_ccaa", "ccaa", "medida"],
    var_name="periodo",
    value_name="var_anual"
)


# Calculo de la columna "fecha"
# extraer año y trimestre
df_ipv_var_anual_ine["ejercicio"] = df_ipv_var_anual_ine["periodo"].str[:4].astype(int)
df_ipv_var_anual_ine["trim"] = df_ipv_var_anual_ine["periodo"].str[-1].astype(int)
# mapa trimestre -> mes
mapa_mes = {
    1: 1,
    2: 4,
    3: 7,
    4: 10
}
# crear fecha
df_ipv_var_anual_ine["fecha"] = pd.to_datetime({
    "year": df_ipv_var_anual_ine["ejercicio"],
    "month": df_ipv_var_anual_ine["trim"].map(mapa_mes),
    "day": 1
})

# Substituir caracteres en blanco en la columna "medida"
df_ipv_var_anual_ine["medida"] = df_ipv_var_anual_ine["medida"].str.strip()
df_ipv_var_anual_ine["medida"] = df_ipv_var_anual_ine["medida"].str.replace(" ", "_")

# Calculo del campo "id"
df_ipv_var_anual_ine["id"] = (
    df_ipv_var_anual_ine["cod_ccaa"].astype(str)
    + "_"
    + df_ipv_var_anual_ine["medida"]
    + "_"
    + df_ipv_var_anual_ine["periodo"]
)

# Ordenar el DF
df_ipv_var_anual_ine = df_ipv_var_anual_ine[['id', 'cod_ccaa', 'ccaa', 'medida', 'periodo', 'fecha', 'ejercicio','trim', 'var_anual']]

# Visualizar los datos
df_ipv_var_anual_ine

In [ ]:
## ***************************************************************************

In [ ]:
## 3. Carga de datos de la variación trimestral del IPV del INE
# Carga del fichero de indices del INE
df_ipv_var_trim_ine_prev = pd.read_excel(indices_file, sheet_name="tabla-25171-vartrim")

## Aplicar transformaciones
# primera columna (la que contiene Nacional, Andalucía, etc.)
col0 = df_ipv_var_trim_ine_prev.columns[0]

# detectar filas cabecera de CCAA:
# aquí usamos una columna numérica cualquiera
mask_ccaa = df_ipv_var_trim_ine_prev["2025T4"].isna()

# crear columna CCAA
df_ipv_var_trim_ine_prev["ccaa"] = df_ipv_var_trim_ine_prev[col0].where(mask_ccaa)

# rellenar hacia abajo
df_ipv_var_trim_ine_prev["ccaa"] = df_ipv_var_trim_ine_prev["ccaa"].ffill()

# mover CCAA al inicio
df_ipv_var_trim_ine_prev.insert(0, "ccaa", df_ipv_var_trim_ine_prev.pop("ccaa"))

# Renombrar columnas
df_ipv_var_trim_ine_prev.rename(
    columns={df_ipv_var_trim_ine_prev.columns[1]: "medida"},
    inplace=True
)

# Eliminar filas con valores NaN
df_ipv_var_trim_ine_prev = df_ipv_var_trim_ine_prev[
    df_ipv_var_trim_ine_prev["ccaa"] != df_ipv_var_trim_ine_prev["medida"]
].reset_index(drop=True)

# Calculo de la columna cod_ccaa
df_ipv_var_trim_ine_prev["cod_ccaa"] = np.where(
    df_ipv_var_trim_ine_prev["ccaa"] == "Nacional",
    99,
    df_ipv_var_trim_ine_prev["ccaa"].str.extract(r"^(\d{2})")[0]
).astype(int)

# mover cod_ccaa al inicio
df_ipv_var_trim_ine_prev.insert(0, "cod_ccaa", df_ipv_var_trim_ine_prev.pop("cod_ccaa"))

# Visualizar los resultados
df_ipv_var_trim_ine_prev

In [ ]:
# Transponer los datos (convertir las columnas de tiempo en filas)
df_ipv_var_trim_ine = df_ipv_var_trim_ine_prev.melt(
    id_vars=["cod_ccaa", "ccaa", "medida"],
    var_name="periodo",
    value_name="var_trim"
)


# Calculo de la columna "fecha"
# extraer año y trimestre
df_ipv_var_trim_ine["ejercicio"] = df_ipv_var_trim_ine["periodo"].str[:4].astype(int)
df_ipv_var_trim_ine["trim"] = df_ipv_var_trim_ine["periodo"].str[-1].astype(int)
# mapa trimestre -> mes
mapa_mes = {
    1: 1,
    2: 4,
    3: 7,
    4: 10
}
# crear fecha
df_ipv_var_trim_ine["fecha"] = pd.to_datetime({
    "year": df_ipv_var_trim_ine["ejercicio"],
    "month": df_ipv_var_trim_ine["trim"].map(mapa_mes),
    "day": 1
})

# Substituir caracteres en blanco en la columna "medida"
df_ipv_var_trim_ine["medida"] = df_ipv_var_trim_ine["medida"].str.strip()
df_ipv_var_trim_ine["medida"] = df_ipv_var_trim_ine["medida"].str.replace(" ", "_")

# Calculo del campo "id"
df_ipv_var_trim_ine["id"] = (
    df_ipv_var_trim_ine["cod_ccaa"].astype(str)
    + "_"
    + df_ipv_var_trim_ine["medida"]
    + "_"
    + df_ipv_var_trim_ine["periodo"]
)

# Ordenar el DF
df_ipv_var_trim_ine = df_ipv_var_trim_ine[['id', 'cod_ccaa', 'ccaa', 'medida', 'periodo', 'fecha', 'ejercicio','trim', 'var_trim']]

# Visualizar los datos
df_ipv_var_trim_ine

In [ ]:
## ***************************************************************************

In [ ]:
# 4. Concatener los 3 DF para obtener el DF final de IPV

# Definición de campos comunes
claves = [
    "id",
    "cod_ccaa",
    "ccaa",
    "medida",
    "periodo",
    "fecha",
    "ejercicio",
    "trim"
]

# merge df1 + df2
df_ipv_ine_final = df_ipv_ine.merge(
    df_ipv_var_anual_ine,
    on=claves,
    how="inner"
)

# merge resultado + df3
df_ipv_ine_final = df_ipv_ine_final.merge(
    df_ipv_var_trim_ine,
    on=claves,
    how="inner"
)

# Visualizar los resultados finales
df_ipv_ine_final

In [ ]:
# Guardar resultados finales de la tabla de df_ipv_ine
df_ipv_ine_final.to_excel("F_ipv_ine_abs_var.xlsx", index=False)
df_ipv_ine_final.to_csv("F_ipv_ine_abs_var.csv", index=False)

In [ ]:
## ***************************************************************************

## ***************************************************************************

In [3]:
# 1. Datos Ponderaciones Vivienda Nueva
# Carga de datos fichero ponderaciones vivienda nueva
df_ipv_ine_pond_vivienda_nueva_prev = pd.read_excel(ponderaciones_file, sheet_name="tabla-25174-nuevavivienda")

# Renombrar columnas
df_ipv_ine_pond_vivienda_nueva_prev.rename(
    columns={df_ipv_ine_pond_vivienda_nueva_prev.columns[0]: "ccaa"},
    inplace=True
)

# Calculo de la columna cod_ccaa
# nombre de la primera columna
col_ccaa = df_ipv_ine_pond_vivienda_nueva_prev.columns[0]
# crear código
df_ipv_ine_pond_vivienda_nueva_prev["cod_ccaa"] = np.where(
    df_ipv_ine_pond_vivienda_nueva_prev[col_ccaa] == "Nacional",
    99,
    df_ipv_ine_pond_vivienda_nueva_prev[col_ccaa].str.extract(r"^(\d{2})")[0]
).astype(int)
# mover la columna al inicio
df_ipv_ine_pond_vivienda_nueva_prev.insert(0, "cod_ccaa", df_ipv_ine_pond_vivienda_nueva_prev.pop("cod_ccaa"))

# Visualizar los resultados
df_ipv_ine_pond_vivienda_nueva_prev

,cod_ccaa,ccaa,2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010,2009,2008
0,99,Nacional,13.642,14.304,15.157,17.763,17.359,13.867,12.703,12.074,12.172,13.618,14.484,16.437,28.6,40.296,49.084,54.673,54.346,46.286
1,1,01 Andalucía,14.393,13.603,16.24,18.996,17.165,12.365,10.131,8.753,9.54,11.834,12.375,14.116,26.181,40.511,52.481,58.213,57.367,48.865
2,2,02 Aragón,16.981,20.282,20.038,23.305,24.124,22.159,21.587,20.129,16.623,18.463,24.181,26.188,38.543,42.896,50.322,57.879,58.572,49.632
3,3,"03 Asturias, Principado de",15.202,13.96,12.305,15.925,15.073,11.819,13.431,12.264,11.56,16.247,19.642,26.089,45.009,57.593,64.66,67.217,62.158,53.908
4,4,"04 Balears, Illes",10.483,13.493,12.147,11.575,10.753,7.541,7.045,7.448,9.527,11.217,10.799,15.46,25.093,33.809,40.361,40.82,40.753,35.76
5,5,05 Canarias,6.803,8.083,8.179,7.985,8.29,7.178,7.951,7.644,8.187,10.792,12.021,13.604,20.725,34.264,46.304,49.63,51.72,49.652
6,6,06 Cantabria,11.575,13.67,13.606,12.162,14.779,12.825,13.188,13.935,15.404,19.317,20.602,24.238,41.473,57.113,65.903,70.69,66.587,54.933
7,7,07 Castilla y León,17.567,18.359,16.806,15.856,14.611,12.243,11.331,11.176,11.988,15.946,17.825,18.407,37.744,50.979,60.943,67.419,68.142,60.332
8,8,08 Castilla - La Mancha,15.553,15.416,13.451,16.465,15.912,15.59,15.396,14.879,16.453,20.092,20.827,20.354,43.256,62.612,71.704,75.576,73.64,66.093
9,9,09 Cataluña,10.429,10.032,10.124,11.922,12.282,9.767,8.711,7.651,8.065,9.084,9.461,10.112,18.79,26.81,31.85,36.85,38.785,33.237


In [6]:
# Transponer los datos (convertir las columnas de tiempo en filas)
df_ipv_ine_pond_vivienda_nueva = df_ipv_ine_pond_vivienda_nueva_prev.melt(
    id_vars=["cod_ccaa", "ccaa"],
    var_name="ejercicio",
    value_name="pond"
)

# Crear columna medida
df_ipv_ine_pond_vivienda_nueva["medida"] = "Vivienda Nueva"

# Inclusión de la columna fecha
df_ipv_ine_pond_vivienda_nueva["fecha"] = pd.to_datetime({
    "year": df_ipv_ine_pond_vivienda_nueva["ejercicio"],
    "month": 12,
    "day": 31
})

# Ordenar las columnas del DF
df_ipv_ine_pond_vivienda_nueva = df_ipv_ine_pond_vivienda_nueva[['cod_ccaa', 'ccaa', 'fecha', 'ejercicio','medida', 'pond']]

# Visualizar los resultados
df_ipv_ine_pond_vivienda_nueva

,cod_ccaa,ccaa,fecha,ejercicio,medida,pond
0,99,Nacional,2025-12-31,2025,Vivienda Nueva,13.642
1,1,01 Andalucía,2025-12-31,2025,Vivienda Nueva,14.393
2,2,02 Aragón,2025-12-31,2025,Vivienda Nueva,16.981
3,3,"03 Asturias, Principado de",2025-12-31,2025,Vivienda Nueva,15.202
4,4,"04 Balears, Illes",2025-12-31,2025,Vivienda Nueva,10.483
...,...,...,...,...,...,...
355,15,"15 Navarra, Comunidad Foral de",2008-12-31,2008,Vivienda Nueva,59.1
356,16,16 País Vasco,2008-12-31,2008,Vivienda Nueva,37.155
357,17,"17 Rioja, La",2008-12-31,2008,Vivienda Nueva,61.39
358,18,18 Ceuta,2008-12-31,2008,Vivienda Nueva,


In [ ]:
## ***************************************************************************

In [7]:
# 2. Datos Ponderaciones Vivienda Segunda Mano
# Carga de datos fichero ponderaciones vivienda segunda mano
df_ipv_ine_pond_segunda_mano_prev = pd.read_excel(ponderaciones_file, sheet_name="tabla-25174-segundamano")

# Renombrar columnas
df_ipv_ine_pond_segunda_mano_prev.rename(
    columns={df_ipv_ine_pond_segunda_mano_prev.columns[0]: "ccaa"},
    inplace=True
)

# Calculo de la columna cod_ccaa
# nombre de la primera columna
col_ccaa = df_ipv_ine_pond_segunda_mano_prev.columns[0]
# crear código
df_ipv_ine_pond_segunda_mano_prev["cod_ccaa"] = np.where(
    df_ipv_ine_pond_segunda_mano_prev[col_ccaa] == "Nacional",
    99,
    df_ipv_ine_pond_segunda_mano_prev[col_ccaa].str.extract(r"^(\d{2})")[0]
).astype(int)
# mover la columna al inicio
df_ipv_ine_pond_segunda_mano_prev.insert(0, "cod_ccaa", df_ipv_ine_pond_segunda_mano_prev.pop("cod_ccaa"))

# Visualizar los resultados
df_ipv_ine_pond_segunda_mano_prev

,cod_ccaa,ccaa,2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010,2009,2008
0,99,Nacional,86.358,85.696,84.843,82.237,82.641,86.133,87.297,87.926,87.828,86.382,85.516,83.563,71.4,59.704,50.916,45.327,45.654,53.714
1,1,01 Andalucía,85.607,86.397,83.76,81.004,82.835,87.635,89.869,91.247,90.46,88.166,87.625,85.884,73.819,59.489,47.519,41.787,42.633,51.135
2,2,02 Aragón,83.019,79.718,79.962,76.695,75.876,77.841,78.413,79.871,83.377,81.537,75.819,73.812,61.457,57.104,49.678,42.121,41.428,50.368
3,3,"03 Asturias, Principado de",84.798,86.04,87.695,84.075,84.927,88.181,86.569,87.736,88.44,83.753,80.358,73.911,54.991,42.407,35.34,32.783,37.842,46.092
4,4,"04 Balears, Illes",89.517,86.507,87.853,88.425,89.247,92.459,92.955,92.552,90.473,88.783,89.201,84.54,74.907,66.191,59.639,59.18,59.247,64.24
5,5,05 Canarias,93.197,91.917,91.821,92.015,91.71,92.822,92.049,92.356,91.813,89.208,87.979,86.396,79.275,65.736,53.696,50.37,48.28,50.348
6,6,06 Cantabria,88.425,86.33,86.394,87.838,85.221,87.175,86.812,86.065,84.596,80.683,79.398,75.762,58.527,42.887,34.097,29.31,33.413,45.067
7,7,07 Castilla y León,82.433,81.641,83.194,84.144,85.389,87.757,88.669,88.824,88.012,84.054,82.175,81.593,62.256,49.021,39.057,32.581,31.858,39.668
8,8,08 Castilla - La Mancha,84.447,84.584,86.549,83.535,84.088,84.41,84.604,85.121,83.547,79.908,79.173,79.646,56.744,37.388,28.296,24.424,26.36,33.907
9,9,09 Cataluña,89.571,89.968,89.876,88.078,87.718,90.233,91.289,92.349,91.935,90.916,90.539,89.888,81.21,73.19,68.15,63.15,61.215,66.763


In [8]:
# Transponer los datos (convertir las columnas de tiempo en filas)
df_ipv_ine_pond_segunda_mano = df_ipv_ine_pond_segunda_mano_prev.melt(
    id_vars=["cod_ccaa", "ccaa"],
    var_name="ejercicio",
    value_name="pond"
)

# Crear columna medida
df_ipv_ine_pond_segunda_mano["medida"] = "Segunda Mano"

# Inclusión de la columna fecha
df_ipv_ine_pond_segunda_mano["fecha"] = pd.to_datetime({
    "year": df_ipv_ine_pond_segunda_mano["ejercicio"],
    "month": 12,
    "day": 31
})

# Ordenar las columnas del DF
df_ipv_ine_pond_segunda_mano = df_ipv_ine_pond_segunda_mano[['cod_ccaa', 'ccaa', 'fecha', 'ejercicio','medida', 'pond']]

# Visualizar los resultados
df_ipv_ine_pond_segunda_mano

,cod_ccaa,ccaa,fecha,ejercicio,medida,pond
0,99,Nacional,2025-12-31,2025,Segunda Mano,86.358
1,1,01 Andalucía,2025-12-31,2025,Segunda Mano,85.607
2,2,02 Aragón,2025-12-31,2025,Segunda Mano,83.019
3,3,"03 Asturias, Principado de",2025-12-31,2025,Segunda Mano,84.798
4,4,"04 Balears, Illes",2025-12-31,2025,Segunda Mano,89.517
...,...,...,...,...,...,...
355,15,"15 Navarra, Comunidad Foral de",2008-12-31,2008,Segunda Mano,40.9
356,16,16 País Vasco,2008-12-31,2008,Segunda Mano,62.845
357,17,"17 Rioja, La",2008-12-31,2008,Segunda Mano,38.61
358,18,18 Ceuta,2008-12-31,2008,Segunda Mano,


In [9]:
# Concatener los dos DF para obtener el resultado final
df_ipv_pond_inefinal = pd.concat([df_ipv_ine_pond_vivienda_nueva, df_ipv_ine_pond_segunda_mano], ignore_index=True)

# Visualizar los resultados
df_ipv_pond_inefinal

,cod_ccaa,ccaa,fecha,ejercicio,medida,pond
0,99,Nacional,2025-12-31,2025,Vivienda Nueva,13.642
1,1,01 Andalucía,2025-12-31,2025,Vivienda Nueva,14.393
2,2,02 Aragón,2025-12-31,2025,Vivienda Nueva,16.981
3,3,"03 Asturias, Principado de",2025-12-31,2025,Vivienda Nueva,15.202
4,4,"04 Balears, Illes",2025-12-31,2025,Vivienda Nueva,10.483
...,...,...,...,...,...,...
715,15,"15 Navarra, Comunidad Foral de",2008-12-31,2008,Segunda Mano,40.9
716,16,16 País Vasco,2008-12-31,2008,Segunda Mano,62.845
717,17,"17 Rioja, La",2008-12-31,2008,Segunda Mano,38.61
718,18,18 Ceuta,2008-12-31,2008,Segunda Mano,


In [10]:
# Guardar resultados finales de la tabla de df_ipv_ine
df_ipv_pond_inefinal.to_excel("F_ipv_ponderaciones_ine.xlsx", index=False)
df_ipv_pond_inefinal.to_csv("F_ipv_ponderaciones_ine.csv", index=False)